# Notebook 2 — Analyzing Graph Features## Clustering Coefficient, Path Length, and Small-WorldednessIn **Notebook 1**, we learned how to build and visualize graphs. Now we'll measure them.Three metrics dominate the analysis of biological networks:1. **Clustering Coefficient (C)** — How densely connected are the neighbors of a node?   - *Biological interpretation*: Measures local processing capacity. In the brain, high clustering means neurons in a local circuit are densely interconnected, enabling efficient local computation.   - *Example*: In cortical columns, neurons processing the same sensory feature are tightly interconnected (high clustering).2. **Path Length (L)** — How many steps does it take to travel between two nodes?   - *Biological interpretation*: Measures global communication efficiency. Short path lengths mean signals can propagate rapidly across brain regions, enabling global integration of information.   - *Example*: A signal traveling from sensory cortex to motor cortex should traverse only a few synaptic steps (not hundreds).3. **Small-Worldedness (σ)** — Can a network have BOTH high clustering AND short path lengths?   - *Biological interpretation*: Small-world networks balance local specialization (clustering) with global integration (short paths). This is the hallmark of biological networks.   - *Citation*: Watts & Strogatz (1998) showed that the brain is small-world; Humphries & Gurney (2008) developed metrics to measure this quantitatively.By the end of this notebook, you'll be able to measure whether a network has these properties—and understand why they matter biologically.

## Setup and ImportsLet's import the standard libraries and the helper function we'll use throughout.

In [ ]:
import networkx as nximport numpy as npimport matplotlib.pyplot as pltimport mathprint("Libraries imported successfully.")

## Helper Function: `analyze_graph()`This helper function computes key metrics for any graph. We'll use it throughout the notebook.

In [ ]:
def analyze_graph(G, name="Graph"):    """Compute and print key graph metrics."""    n = G.number_of_nodes()    e = G.number_of_edges()    max_edges = n * (n - 1) / 2    density = e / max_edges if max_edges > 0 else 0    cc = nx.average_clustering(G)        # For disconnected graphs, compute largest component's path length    if nx.is_connected(G):        apl = nx.average_shortest_path_length(G)    else:        largest_cc = max(nx.connected_components(G), key=len)        G_sub = G.subgraph(largest_cc).copy()        if len(G_sub) > 1:            apl = nx.average_shortest_path_length(G_sub)        else:            apl = 0        print(f"\n--- {name} ---")    print(f"Nodes: {n}, Edges: {e}")    print(f"Density: {density:.4f}")    print(f"Avg Clustering Coefficient: {cc:.4f}")    print(f"Avg Path Length (largest component): {apl:.4f}")        return {        "name": name,        "n": n,        "edges": e,        "density": density,        "cc": cc,        "apl": apl    }print("Helper function defined: analyze_graph()")

---# Part 1: Clustering Coefficients## What is Clustering?The **clustering coefficient** measures how connected a node's neighbors are to each other. It answers the question:**"If two of my neighbors are both connected to me, are they also connected to each other?"**In the brain, this translates to: "Do neurons in the same local circuit form tight groups (cliques), or are they loosely connected?"High clustering means local redundancy—multiple paths for information within a small region. Low clustering means information flows in more linear, feed-forward ways.

## Step 1: Identifying a Node's NeighborsIn graph theory, a **neighbor** of node $v$ is another node $u$ that is directly connected to $v$ by an edge.**Example**: If node $A$ is connected to nodes $B$ and $C$, then $B$ and $C$ are the neighbors of $A$.The number of neighbors a node has is called its **degree**.

In [ ]:
# Create a sample graph to use throughout the clustering coefficient sectionG = nx.Graph()G.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'D')])# Position nodes using a layout for consistent visualizationpos = nx.spring_layout(G, seed=42)# Visualize: highlight node A and its neighbors B and Cfig, ax = plt.subplots(1, 1, figsize=(8, 6))# Draw the entire graph with all nodes in light greynx.draw(G, pos, with_labels=True, node_size=500, node_color='lightgrey',         font_weight='bold', edge_color='grey', ax=ax)# Highlight node A (focus) in rednx.draw_networkx_nodes(G, pos, nodelist=['A'], node_color='salmon',                        node_size=700, edgecolors='black', linewidths=2, ax=ax)# Highlight neighbors B and C in skybluenx.draw_networkx_nodes(G, pos, nodelist=['B', 'C'], node_color='skyblue',                        node_size=700, edgecolors='black', linewidths=2, ax=ax)# Highlight node D in grey (not a neighbor)nx.draw_networkx_nodes(G, pos, nodelist=['D'], node_color='lightgrey',                        node_size=700, edgecolors='black', linewidths=2, ax=ax)ax.set_title("Step 1: Identifying Node A's Neighbors (B and C)", fontsize=14, fontweight='bold')ax.legend(['Node A (focus)', 'Neighbors of A', 'Other nodes'], loc='upper left', fontsize=10)plt.tight_layout()plt.show()# Calculate and print the degree of node Adegree_A = G.degree('A')print(f"Degree of node A: {degree_A}")print(f"Neighbors of node A: {list(G.neighbors('A'))}")

## Step 2: Counting Connections Among NeighborsNow, focus on the neighbors of node $A$ (nodes $B$ and $C$). The question is:**"How many edges exist between $B$ and $C$?"**If $B$ and $C$ are connected, they form a **triangle** with $A$: a "clique" of 3 fully connected nodes.

In [ ]:
# Visualize: focus on node A and the connections between its neighborsfig, ax = plt.subplots(1, 1, figsize=(8, 6))nx.draw(G, pos, with_labels=True, node_size=500, node_color='lightgrey',         font_weight='bold', edge_color='grey', ax=ax)# Highlight node Anx.draw_networkx_nodes(G, pos, nodelist=['A'], node_color='salmon',                        node_size=700, edgecolors='black', linewidths=2, ax=ax)# Highlight neighbors B and Cnx.draw_networkx_nodes(G, pos, nodelist=['B', 'C'], node_color='skyblue',                        node_size=700, edgecolors='black', linewidths=2, ax=ax)# Highlight the edges that form the cliqueclique_edges = [('A', 'B'), ('A', 'C'), ('B', 'C')]nx.draw_networkx_edges(G, pos, edgelist=clique_edges, edge_color='red',                        width=3, ax=ax)ax.set_title("Step 2: Connections Among A's Neighbors (B and C form a clique with A)",              fontsize=14, fontweight='bold')plt.tight_layout()plt.show()# Count actual edges between neighbors of Aneighbors_of_A = list(G.neighbors('A'))actual_edges = 0for i, u in enumerate(neighbors_of_A):    for v in neighbors_of_A[i+1:]:        if G.has_edge(u, v):            actual_edges += 1print(f"Neighbors of A: {neighbors_of_A}")print(f"Number of actual edges between A's neighbors: {actual_edges}")

## Step 3: The Clustering Coefficient FormulaThe clustering coefficient of a node measures how close its neighbors are to forming a complete clique (where every possible edge exists).**Formula**: For a node $A$ with $k$ neighbors:$$C_A = \frac{\text{Number of actual edges between neighbors}}{\text{Maximum possible edges between neighbors}}$$The maximum number of possible edges between $k$ neighbors is $\binom{k}{2} = \frac{k(k-1)}{2}$.So:$$C_A = \frac{2E}{k(k-1)}$$where $E$ is the number of actual edges between neighbors, and the factor of 2 accounts for undirected edges.

In [ ]:
# Step-by-step calculation for node Aneighbors_A = list(G.neighbors('A'))k_A = len(neighbors_A)# Count actual edges between neighborsactual_edges_A = 0edges_list = []for i, u in enumerate(neighbors_A):    for v in neighbors_A[i+1:]:        if G.has_edge(u, v):            actual_edges_A += 1            edges_list.append((u, v))# Maximum possible edges between neighborsmax_edges_A = k_A * (k_A - 1) / 2# Calculate clustering coefficientclustering_A = actual_edges_A / max_edges_A if max_edges_A > 0 else 0print(f"\nClustering Coefficient Calculation for Node A:")print(f"  Degree (k): {k_A}")print(f"  Neighbors: {neighbors_A}")print(f"  Actual edges between neighbors: {actual_edges_A} (edge(s): {edges_list})")print(f"  Maximum possible edges: {int(max_edges_A)}")print(f"  Clustering Coefficient = {actual_edges_A} / {int(max_edges_A)} = {clustering_A:.2f}")print(f"\nInterpretation: {clustering_A*100:.0f}% of the possible edges among A's neighbors exist.")

## Computing Clustering Coefficient for All NodesNow let's calculate the clustering coefficient for each node in the graph and visualize them side by side.

In [ ]:
# Calculate clustering coefficient for each nodeclustering_coeffs = nx.clustering(G)# Calculate the average clustering coefficientavg_clustering_coeff = sum(clustering_coeffs.values()) / len(G)# Create subplots for each nodefig, axes = plt.subplots(1, 4, figsize=(16, 4))pos = nx.circular_layout(G)for i, (node, cc) in enumerate(clustering_coeffs.items()):    ax = axes[i]        # Draw the graph    nx.draw(G, pos, with_labels=True, node_size=500, node_color='lightgrey',             edge_color='grey', font_weight='bold', ax=ax)        # Highlight the current node    nx.draw_networkx_nodes(G, pos, nodelist=[node], node_color='salmon',                            node_size=700, ax=ax)        ax.set_title(f"Node {node}\nCC = {cc:.2f}", fontsize=12, fontweight='bold')    ax.axis('off')plt.suptitle("Clustering Coefficients for Each Node", fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()# Print summaryprint(f"\nClustering Coefficients:")for node, cc in clustering_coeffs.items():    print(f"  Node {node}: {cc:.2f}")print(f"\nAverage Clustering Coefficient (C_avg): {avg_clustering_coeff:.2f}")

## Global Clustering Coefficient (Transitivity)In addition to the **average clustering coefficient**, there's another way to measure clustering at the graph level: **transitivity** (also called the **global clustering coefficient**).Transitivity measures the ratio of **triangles** (3-node cliques) to **connected triples** (3 nodes where at least one is connected to the other two).**Formula**:$$T = \frac{3 \times \text{Number of Triangles}}{\text{Number of Connected Triples}}$$The factor of 3 accounts for the fact that each triangle contains three edges (triples).**Biological interpretation**: A high transitivity value indicates that the network has many tightly connected clusters (modules). This is characteristic of biological networks, where specialized regions (like cortical columns) are densely interconnected internally.

In [ ]:
# Calculate both clustering metricsaverage_cc = nx.average_clustering(G)transitivity = nx.transitivity(G)print(f"\nGlobal Clustering Metrics for the Graph:")print(f"  Average Clustering Coefficient: {average_cc:.4f}")print(f"  Transitivity (Global Clustering Coefficient): {transitivity:.4f}")# Interpretationif transitivity > 0.3:    print(f"\n  Interpretation: High clustering! The network has many triangles (cliques).")elif transitivity > 0.1:    print(f"\n  Interpretation: Moderate clustering. Some local structure exists.")else:    print(f"\n  Interpretation: Low clustering. The network is relatively tree-like or random.")

---# Part 2: Path Length## What is Path Length?The **path length** between two nodes is the minimum number of edges you must traverse to travel from one node to the other. It's also called the **shortest path** or **geodesic distance**.**Biological interpretation**: In the brain, path length measures how many synaptic steps are needed for a signal to travel from one region to another. Short path lengths enable rapid global communication; long path lengths slow information transfer.

## Shortest vs. Longer PathsLet's visualize the difference between the shortest path and alternative (longer) paths between two nodes.

In [ ]:
# Create a slightly more complex graph for path length demonstrationsG_paths = nx.Graph()edges = [('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'D'), ('D', 'E'), ('A', 'E')]G_paths.add_edges_from(edges)# Define positions for consistent visualizationpos = nx.circular_layout(G_paths)fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Plot 1: Shortest path between A and Dax = axes[0]nx.draw(G_paths, pos, with_labels=True, node_size=500, node_color='lightgrey',         edge_color='grey', font_weight='bold', ax=ax)shortest_path_AD = nx.shortest_path(G_paths, 'A', 'D')path_edges_AD = list(zip(shortest_path_AD, shortest_path_AD[1:]))nx.draw_networkx_nodes(G_paths, pos, nodelist=shortest_path_AD, node_color='salmon',                        node_size=700, ax=ax)nx.draw_networkx_edges(G_paths, pos, edgelist=path_edges_AD, edge_color='red',                        width=3, ax=ax)ax.set_title(f"Shortest Path A→D: {' → '.join(shortest_path_AD)}\nLength: {len(path_edges_AD)} edges",              fontsize=12, fontweight='bold')ax.axis('off')# Plot 2: A longer path between A and D (if it exists)ax = axes[1]nx.draw(G_paths, pos, with_labels=True, node_size=500, node_color='lightgrey',         edge_color='grey', font_weight='bold', ax=ax)# Try to find a longer path manually: A -> B -> C -> Dlonger_path_AD = ['A', 'B', 'C', 'D']longer_path_edges = list(zip(longer_path_AD, longer_path_AD[1:]))nx.draw_networkx_nodes(G_paths, pos, nodelist=longer_path_AD, node_color='skyblue',                        node_size=700, ax=ax)nx.draw_networkx_edges(G_paths, pos, edgelist=longer_path_edges, edge_color='blue',                        width=3, ax=ax)ax.set_title(f"Alternative Path A→D: {' → '.join(longer_path_AD)}\nLength: {len(longer_path_edges)} edges",              fontsize=12, fontweight='bold')ax.axis('off')plt.tight_layout()plt.show()# Calculate specific path lengthssp_length = nx.shortest_path_length(G_paths, 'A', 'D')print(f"\nPath Length Analysis (A to D):")print(f"  Shortest path: {' → '.join(shortest_path_AD)}, Length = {len(path_edges_AD)} edges")print(f"  Alternative path: {' → '.join(longer_path_AD)}, Length = {len(longer_path_edges)} edges")print(f"  Using NetworkX: shortest_path_length(A, D) = {sp_length}")

## Average Path Length (APL)The **Average Path Length** of a graph is the average of the shortest path lengths across all pairs of nodes.**Formula**:$$L = \frac{1}{N(N-1)} \sum_{i \neq j} d(i,j)$$where $N$ is the number of nodes, and $d(i,j)$ is the shortest path length between nodes $i$ and $j$.**Biological interpretation**: Short average path lengths indicate that the brain can rapidly integrate information from distant regions. This is a hallmark of biologically plausible networks.

## Computing APL: All-Pairs Shortest PathsLet's compute the shortest path between every pair of nodes in the graph and visualize them.

In [ ]:
# Calculate all shortest pathspath_lengths = dict(nx.shortest_path_length(G_paths))avg_path_length = nx.average_shortest_path_length(G_paths)# Count the number of unique node pairsnum_nodes = len(G_paths.nodes())num_pairs = num_nodes * (num_nodes - 1) // 2num_cols = 3num_rows = (num_pairs + num_cols - 1) // num_colsfig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 10))axes = axes.flatten()# Plot indexplot_idx = 0pos = nx.circular_layout(G_paths)# Iterate over all node pairs to plot shortest pathsfor i, node1 in enumerate(G_paths.nodes()):    for j, node2 in enumerate(G_paths.nodes()):        if j <= i:            continue  # Skip duplicate pairs and self-pairs        ax = axes[plot_idx]        nx.draw(G_paths, pos, with_labels=True, edge_color='grey',                node_color='lightgrey', node_size=500, ax=ax)        # Highlight the shortest path        shortest_path = nx.shortest_path(G_paths, source=node1, target=node2)        path_edges = list(zip(shortest_path, shortest_path[1:]))        nx.draw_networkx_nodes(G_paths, pos, nodelist=shortest_path,                               node_color='salmon', node_size=600, ax=ax)        nx.draw_networkx_edges(G_paths, pos, edgelist=path_edges,                               edge_color='red', width=2, ax=ax)        ax.set_title(f"{node1} to {node2}, Length: {path_lengths[node1][node2]}",                     fontsize=11, fontweight='bold')        ax.axis('off')        plot_idx += 1# Hide extra subplotsfor idx in range(plot_idx, len(axes)):    axes[idx].axis('off')fig.suptitle(f"All-Pairs Shortest Paths\nAverage Path Length: {avg_path_length:.2f}",             fontsize=14, fontweight='bold', y=0.98)plt.tight_layout()plt.show()

## Manual APL CalculationLet's calculate the average path length step-by-step to understand the formula.

In [ ]:
# Step 1: Extract all pairwise distancesdistances = []node_list = list(G_paths.nodes())for i, node1 in enumerate(node_list):    for node2 in node_list[i+1:]:        dist = path_lengths[node1][node2]        distances.append(dist)        print(f"  d({node1}, {node2}) = {dist}")# Step 2: Sum and averagetotal_distance = sum(distances)num_pairs = len(distances)manual_apl = total_distance / num_pairsprint(f"\nAverage Path Length Calculation:")print(f"  Sum of all pairwise distances: {total_distance}")print(f"  Number of pairs: {num_pairs}")print(f"  Average Path Length = {total_distance} / {num_pairs} = {manual_apl:.4f}")print(f"\nUsing NetworkX function: {avg_path_length:.4f}")print(f"Match: {abs(manual_apl - avg_path_length) < 1e-6}")

## Computing APL for a Real GraphNow let's generate a larger graph and compute its clustering coefficient and average path length.

In [ ]:
# Generate an Erdos-Renyi random graphG_er = nx.erdos_renyi_graph(20, 0.15, seed=42)# Ensure the graph is connected (for valid APL)while not nx.is_connected(G_er):    G_er = nx.erdos_renyi_graph(20, 0.15, seed=np.random.randint(0, 10000))# Analyze the graphresult = analyze_graph(G_er, "Erdos-Renyi (n=20, p=0.15)")# Visualizefig, ax = plt.subplots(1, 1, figsize=(10, 8))pos = nx.spring_layout(G_er, seed=42)nx.draw(G_er, pos, with_labels=True, node_color='skyblue', node_size=500,        edge_color='grey', font_weight='bold', ax=ax)ax.set_title(f"Erdos-Renyi Graph\nC_avg = {result['cc']:.4f}, L = {result['apl']:.4f}",             fontsize=12, fontweight='bold')plt.tight_layout()plt.show()

---# Part 3: Small-Worldedness## The Small-World PhenomenonHere's a paradox: A **lattice** (like a grid of neurons in a cortical column) has **high clustering** (neighbors of a node are neighbors of each other), but **long average path length** (you have to travel far to reach distant nodes).A **random graph** has **low clustering** (edges are random, so neighbors are unlikely to be neighbors of each other), but **short average path length** (random edges create "shortcuts" across the network).**But biological networks are different**: They have **BOTH high clustering AND short path length**.This special property is called **small-worldedness**. A small-world network combines the local structure (clustering) of a lattice with the global efficiency (short paths) of a random graph.**Biological relevance**: The brain is small-world. This allows it to balance:- **Local specialization**: Cortical columns perform specialized computations (high clustering)- **Global integration**: Information travels rapidly across brain regions (short paths)*Citation*: Watts & Strogatz (1998) described small-world networks and showed that the C. elegans nervous system is small-world. Later work showed the same for mammalian brains.

## The Small-World Coefficient (σ)To quantify small-worldedness, we use the **small-world coefficient** (Humphries & Gurney 2008):$$\sigma = \frac{C / C_{\text{random}}}{L / L_{\text{random}}}$$where:- $C$ = average clustering coefficient of the network- $L$ = average path length of the network- $C_{\text{random}}$ = average clustering coefficient of a random graph with the same size and density- $L_{\text{random}}$ = average path length of a random graph with the same size and density**Interpretation**:- $\sigma > 1$: Small-world (high clustering relative to random graphs, short paths relative to lattices)- $\sigma \approx 1$: Not small-world (properties similar to random graphs)- $\sigma < 1$: Lattice-like (high clustering but long paths)**Biological networks**: Typical brain networks have $\sigma$ between 1.5 and 5, indicating strong small-world properties.

## Computing the Small-World CoefficientLet's implement the calculation and test it on different types of networks.

In [ ]:
def small_world_coefficient(G, n_random=10, seed=None):    """    Calculate the small-world coefficient sigma.        sigma = (C / C_rand) / (L / L_rand)    sigma > 1 indicates small-world properties.        Parameters:    -----------    G : networkx.Graph        The network to analyze    n_random : int        Number of random graphs to generate for comparison    seed : int or None        Random seed for reproducibility        Returns:    --------    sigma : float        The small-world coefficient    """    rng = np.random.default_rng(seed)        # Get metrics for the network    C = nx.average_clustering(G)    if not nx.is_connected(G):        largest_cc = max(nx.connected_components(G), key=len)        G_connected = G.subgraph(largest_cc).copy()    else:        G_connected = G        L = nx.average_shortest_path_length(G_connected)        # Generate random graphs with the same size and edge density    C_rand_list = []    L_rand_list = []    n = G.number_of_nodes()    m = G.number_of_edges()    p = 2 * m / (n * (n - 1))  # Connection probability        for i in range(n_random):        # Generate random graph        G_rand = nx.erdos_renyi_graph(n, p, seed=rng.integers(0, 100000))                # Ensure connectivity for path length calculation        attempt = 0        while not nx.is_connected(G_rand) and attempt < 10:            G_rand = nx.erdos_renyi_graph(n, p, seed=rng.integers(0, 100000))            attempt += 1                if nx.is_connected(G_rand):            C_rand_list.append(nx.average_clustering(G_rand))            L_rand_list.append(nx.average_shortest_path_length(G_rand))        # Compute averages    C_rand = np.mean(C_rand_list) if C_rand_list else p    L_rand = np.mean(L_rand_list) if L_rand_list else np.log(n)        # Compute sigma    sigma = (C / C_rand) / (L / L_rand) if C_rand > 0 and L_rand > 0 else 0        print(f"\nSmall-World Coefficient Analysis:")    print(f"  Network C: {C:.4f}, Random C: {C_rand:.4f}, Ratio: {C/C_rand:.4f}")    print(f"  Network L: {L:.4f}, Random L: {L_rand:.4f}, Ratio: {L/L_rand:.4f}")    print(f"  Small-world coefficient (sigma): {sigma:.4f}")    print(f"  Small-world? {'Yes (sigma > 1)' if sigma > 1 else 'No (sigma <= 1)'}")        return sigma, C, L, C_rand, L_randprint("Function defined: small_world_coefficient()")

## Testing Small-Worldedness on Different Network TypesLet's compare three types of networks: lattice, random, and small-world.

In [ ]:
n = 50  # Number of nodes# 1. Ring lattice (high clustering, high path length)print("\n" + "="*70)print("Network 1: Ring Lattice (p=0.0)")print("="*70)G_lattice = nx.watts_strogatz_graph(n, k=4, p=0.0, seed=42)sigma_lattice, C_lattice, L_lattice, C_rand_lattice, L_rand_lattice = small_world_coefficient(    G_lattice, n_random=5, seed=42)# 2. Random graph (low clustering, low path length)print("\n" + "="*70)print("Network 2: Random Graph (ER)")print("="*70)G_random = nx.erdos_renyi_graph(n, p=0.15, seed=42)while not nx.is_connected(G_random):    G_random = nx.erdos_renyi_graph(n, p=0.15, seed=np.random.randint(0, 10000))sigma_random, C_random, L_random, C_rand_random, L_rand_random = small_world_coefficient(    G_random, n_random=5, seed=42)# 3. Small-world network (high clustering, low path length)print("\n" + "="*70)print("Network 3: Small-World (Watts-Strogatz, p=0.3)")print("="*70)G_smallworld = nx.watts_strogatz_graph(n, k=4, p=0.3, seed=42)sigma_sw, C_sw, L_sw, C_rand_sw, L_rand_sw = small_world_coefficient(    G_smallworld, n_random=5, seed=42)

## Visualizing the Three Network Types

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))# Plot 1: Latticeax = axes[0]pos_lattice = nx.circular_layout(G_lattice)nx.draw(G_lattice, pos_lattice, with_labels=False, node_color='lightcoral',        node_size=100, edge_color='grey', alpha=0.3, ax=ax)ax.set_title(f"Lattice (p=0.0)\nC={C_lattice:.3f}, L={L_lattice:.3f}\nsigma={sigma_lattice:.2f}",             fontsize=12, fontweight='bold')# Plot 2: Randomax = axes[1]pos_random = nx.spring_layout(G_random, seed=42)nx.draw(G_random, pos_random, with_labels=False, node_color='lightgreen',        node_size=100, edge_color='grey', alpha=0.3, ax=ax)ax.set_title(f"Random (ER)\nC={C_random:.3f}, L={L_random:.3f}\nsigma={sigma_random:.2f}",             fontsize=12, fontweight='bold')# Plot 3: Small-Worldax = axes[2]pos_sw = nx.circular_layout(G_smallworld)nx.draw(G_smallworld, pos_sw, with_labels=False, node_color='skyblue',        node_size=100, edge_color='grey', alpha=0.3, ax=ax)ax.set_title(f"Small-World (p=0.3)\nC={C_sw:.3f}, L={L_sw:.3f}\nsigma={sigma_sw:.2f}",             fontsize=12, fontweight='bold')for ax in axes:    ax.axis('off')plt.suptitle("Three Network Types: Lattice, Random, and Small-World",             fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

## The Watts-Strogatz Phase Transition: From Lattice to RandomNow let's see how clustering and path length change as we increase the rewiring probability p in a Watts-Strogatz graph. This demonstrates the classic small-world transition.

In [ ]:
# Vary the rewiring probability and compute metricsp_values = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]results = []for p in p_values:    G_ws = nx.watts_strogatz_graph(50, k=4, p=p, seed=42)    c = nx.average_clustering(G_ws)    l = nx.average_shortest_path_length(G_ws)    results.append({'p': p, 'C': c, 'L': l})    print(f"p={p:.2f}: C={c:.4f}, L={l:.4f}")# Extract values for plottingps = [r['p'] for r in results]cs = [r['C'] for r in results]ls = [r['L'] for r in results]# Normalize for comparison (since scales differ)c_norm = [c / cs[0] for c in cs]l_norm = [l / ls[0] for l in ls]# Plotfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Left: Normalized valuesax = axes[0]ax.plot(ps, c_norm, 'o-', linewidth=2, markersize=8, label='Clustering (normalized)', color='red')ax.plot(ps, l_norm, 's-', linewidth=2, markersize=8, label='Path Length (normalized)', color='blue')ax.axvline(x=0.3, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Sweet spot (p ~ 0.3)')ax.set_xlabel('Rewiring Probability (p)', fontsize=12, fontweight='bold')ax.set_ylabel('Normalized Value (relative to p=0)', fontsize=12, fontweight='bold')ax.set_title('Watts-Strogatz Phase Transition (Normalized)', fontsize=12, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)# Right: Absolute valuesax = axes[1]ax2 = ax.twinx()line1 = ax.plot(ps, cs, 'o-', linewidth=2, markersize=8, label='Clustering Coefficient', color='red')line2 = ax2.plot(ps, ls, 's-', linewidth=2, markersize=8, label='Average Path Length', color='blue')ax.axvline(x=0.3, color='green', linestyle='--', linewidth=2, alpha=0.5)ax.set_xlabel('Rewiring Probability (p)', fontsize=12, fontweight='bold')ax.set_ylabel('Clustering Coefficient', fontsize=12, fontweight='bold', color='red')ax2.set_ylabel('Average Path Length', fontsize=12, fontweight='bold', color='blue')ax.set_title('Watts-Strogatz Phase Transition (Absolute Values)', fontsize=12, fontweight='bold')ax.tick_params(axis='y', labelcolor='red')ax2.tick_params(axis='y', labelcolor='blue')ax.grid(True, alpha=0.3)# Combined legendlines = line1 + line2labels = [l.get_label() for l in lines]ax.legend(lines, labels, loc='center right', fontsize=11)plt.tight_layout()plt.show()print(f"\nInterpretation:")print(f"  At p=0: High clustering, high path length (lattice)")print(f"  At p~0.3: HIGH clustering AND short path length (SMALL-WORLD)")print(f"  At p=1: Low clustering, low path length (random)")

## Biological Network Properties: A ChecklistNow that we can measure clustering, path length, and small-worldedness, let's see how these metrics relate to the biological properties we care about.| Property | What we can measure now | What we still need | Which Notebook? ||----------|-------------------------|-------------------|--------------------|| **Local specialization (clustering)** | Clustering coefficient (C) | ✓ We have this | *This notebook* || **Global integration (path length)** | Average Path Length (L) | ✓ We have this | *This notebook* || **Small-worldedness** | Small-world coefficient (σ) | ✓ We have this | *This notebook* || **Modularity (communities)** | Partial (from clustering) | Full community detection | **Notebook 7** || **Hierarchical organization** | None yet | Hierarchical clustering | **Notebook 7** || **Directionality** | None yet (our graphs are undirected) | Directed edges | **Future** || **Weighted structure** | None yet (all edges equal) | Edge weights (importance) | **Future** || **Sparsity** | Density (inverse of sparsity) | ✓ We have this | *This notebook* |**Key insight**: Clustering coefficient, path length, and small-worldedness are foundational metrics. They let us evaluate whether generated networks (Notebooks 3-7) are biologically plausible.

---## Key Takeaways### What We've Learned1. **Clustering Coefficient** measures local structure   - **Definition**: The fraction of possible edges among a node's neighbors that actually exist   - **Biological meaning**: Measures local processing capacity and module coherence   - **Range**: 0 (no neighbors are connected) to 1 (all neighbors form a complete clique)   - **Brain relevance**: Cortical columns have high clustering; long-range connections have lower clustering2. **Average Path Length** measures global efficiency   - **Definition**: Average shortest path across all pairs of nodes   - **Biological meaning**: How quickly signals propagate across the network   - **Brain relevance**: Mammalian brains have short path lengths (2-4 synaptic steps between distant regions)   - **Trade-off**: Shorter paths require more long-range connections (metabolically expensive)3. **Small-Worldedness** combines both properties   - **Definition**: High clustering + short path length (captured by σ > 1)   - **Biological meaning**: Balance between local specialization and global integration   - **Brain relevance**: The brain IS small-world (σ typically 1.5–5)   - **Why it matters**: Small-world networks are robust, efficient, and support both specialized and distributed computation4. **These metrics are our evaluation tools**   - When we generate networks in Notebooks 3–7, we'll measure these three metrics   - Networks that don't match biological values are less plausible   - The "gold standard" is networks that are small-world AND sparse### Conceptual Summary- **Lattice**: High C, high L → Not small-world (σ < 1)- **Random**: Low C, low L → Not small-world (σ ≈ 1)- **Small-world**: High C, low L → IS small-world (σ > 1) ← **This is what biology achieves**### Why Biological Networks Are Small-WorldBiological constraints favor small-world structure:1. **Space is limited**: Brains are packed in skulls; long-distance connections are metabolically expensive2. **Locality matters**: Neurons naturally cluster by proximity, creating local modules with high clustering3. **But global integration is essential**: The brain must bind distributed information (vision + sound + proprioception → unified perception)4. **Solution**: Strategic long-range connections (called "shortcuts" in network theory) create short paths without requiring dense global connectivityThis is what evolution discovered: **small-world networks**.

---## What's Next: Notebook 3### Generating Sparse NetworksNow that we know *what* to measure (clustering, path length, small-worldedness), the next question is:**How do we generate networks with these properties?**In **Notebook 3: Generating Sparse Networks (Techniques)**, we'll explore:1. **Erdős-Rényi graphs** — The simplest random model; our baseline2. **Watts-Strogatz graphs** — How rewiring creates small-world structure3. **Barabási-Albert graphs** — How preferential attachment creates hub-like structure4. **Spatial/Geometric graphs** — How embedding networks in space naturally creates biological structure5. **Comparison** — Which method best matches biological networks?By the end of Notebook 3, you'll understand the trade-offs between different generation methods and why some are more biologically plausible than others.**Teaser**: Spatial networks, where connection probability decreases with distance, naturally produce:- High clustering (nearby nodes connect)- Short path lengths (distance constraints prevent excessive sparsity)- Emergent modularity (clusters form naturally)- Weighted structure (distance → weight)They're the closest match to biological networks among simple generative models.